 # Comprehensive Example Notebook

 This notebook demonstrates how to use the Prompt classes from `agent_platform_core`
 to create a rich conversation flow that includes:

 - **System Instructions** (defining the AI's role/behavior)
 - **User and Agent Messages** with multiple content types:
   - Text (`PromptTextContent`)
   - Image (`PromptImageContent`)
   - Audio (`PromptAudioContent`)
 - **Tool Usage** (`PromptToolUseContent`) and **Tool Results** (`PromptToolResultContent`)
 - **Conversation Conversion** to Gemini format
 - **Model Inference** via Google's Generative AI (Gemini)

 ## 1) Install Dependencies & Set Up Environment
 Run the following cell to make sure you have dependencies installed (common example notebook dependencies and our `agent_platform_core` package with the `providers-google` extra set).
 Make sure your environment variables (e.g., `GEMINI_API_KEY`) are set accordingly before continuing.

In [1]:
%%capture
%pip install -r requirements-common.txt -e "..[providers-google]"

## 2) Notebook Utilities & Environment Checks

In this section, we import required utilities and confirm our environment is ready
(e.g., `GEMINI_API_KEY` is set).

In [1]:
import os

# Utility for nice markdown display
from IPython.display import Markdown
from IPython.display import display as ip_display

# Generic utils
from examples.utils import setup_notebook

# Check environment setup
if not setup_notebook(required_keys=["GEMINI_API_KEY"]):
    raise OSError("Please set up required environment variables")

# Configure Gemini-specific provider package
import google.generativeai as genai

# Import Gemini-specific provider conversion
from agent_platform_core.models.providers.google import (
    convert_prompt_to_gemini_dict,
)

# Import our generic prompt types
from agent_platform_core.prompts import (
    Prompt,
    PromptAgentMessage,
    PromptAudioContent,
    PromptImageContent,
    PromptTextContent,
    PromptToolResultContent,
    PromptToolUseContent,
    PromptUserMessage,
)

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

## 3) Construct a Complex Prompt

Below, we build a `Prompt` that:
1. Defines a **system instruction** ("You are a helpful AI agent...").
2. Contains multiple **user** and **agent** messages with different content types:
    - **Text**  
    - **Audio** (URL-based)  
    - **Image** (URL-based example)  
    - **Tool Usage** (the agent requesting a 'Weather' tool)  
    - **Tool Result** (simulating the returned weather data)  
3. Sets a **temperature** of 0.0 and a seed=42 to bias towards deterministic responses.

In [2]:
prompt = Prompt(
    system_instruction=(
        "You are a helpful AI assistant. Provide concise, accurate answers "
        "using any tool calls or analysis as needed."
    ),
    messages=[
        # ---------------------------------------------------------
        # 1) User starts with a text message
        # ---------------------------------------------------------
        PromptUserMessage(
            content=[
                PromptTextContent(
                    text=(
                        "Hello AI! I'd like to know the "
                        "current weather in New York City."
                    ),
                ),
            ],
        ),

        # ---------------------------------------------------------
        # 2) Agent message proposing a tool call to gather weather
        # ---------------------------------------------------------
        PromptAgentMessage(
            content=[
                PromptTextContent(
                    text="Sure, I can help with that! Let me check my weather tool...",
                ),
                PromptToolUseContent(
                    tool_name="WeatherTool",
                    tool_call_id="tool_call_12345",
                    tool_input_raw='{"location": "New York City", "units": "imperial"}',
                ),
            ],
        ),

        # ---------------------------------------------------------
        # 3) User message returning the tool result
        #    (In a real scenario, your application or agent orchestrator
        #    would fill this in after calling the tool. We're simulating it.)
        # ---------------------------------------------------------
        PromptUserMessage(
            content=[
                PromptToolResultContent(
                    tool_name="WeatherTool",
                    tool_call_id="tool_call_12345",
                    content=[
                        PromptTextContent(
                            text="The weather in NYC is 72F, partly cloudy.",
                        ),
                    ],
                    is_error=False,
                ),
            ],
        ),

        # ---------------------------------------------------------
        # 4) Agent responds to user with the final weather summary
        # ---------------------------------------------------------
        PromptAgentMessage(
            content=[
                PromptTextContent(
                    text=(
                        "The current weather in NYC is 72°F"
                        " and partly cloudy. Need anything else?"
                    ),
                ),
            ],
        ),

        # ---------------------------------------------------------
        # 5) User provides an audio snippet they'd like summarized
        # ---------------------------------------------------------
        PromptUserMessage(
            content=[
                PromptTextContent(
                    text="Yes. Could you also summarize this short audio file for me?",
                ),
                # For demonstration, using a generic mp3 file URL
                PromptAudioContent(
                    mime_type="audio/mp3",
                    value="https://www.voxtab.com/downloads/audio-files/VOXTAB_Business_audio.mp3",
                    sub_type="url",
                ),
            ],
        ),

        # ---------------------------------------------------------
        # 6) Agent acknowledging the audio
        # ---------------------------------------------------------
        PromptAgentMessage(
            content=[
                PromptTextContent(
                    text="Alright, I'm analyzing the audio file now...",
                ),
            ],
        ),

        # ---------------------------------------------------------
        # 7) User shares an image they'd like described
        # ---------------------------------------------------------
        PromptUserMessage(
            content=[
                PromptTextContent(
                    text="Also, here's an image I'd like you to describe.",
                ),
                PromptImageContent(
                    mime_type="image/png",
                    value=(
                        "https://upload.wikimedia.org/wikipedia/commons/thumb"
                        "/0/00/Brooklyn_Bridge_Manhattan.jpg/1920px-Brooklyn_Bridge_Manhattan.jpg"
                    ),
                    sub_type="url",
                    detail="high_res",
                ),
            ],
        ),

        # ---------------------------------------------------------
        # 8) Agent clarifies instructions
        # ---------------------------------------------------------
        PromptAgentMessage(
            content=[
                PromptTextContent(
                    text=(
                        "So, I'll summarize the audio and describe the image."
                        "Any specific formatting requests?"
                    ),
                ),
            ],
        ),

        # ---------------------------------------------------------
        # 9) User final instructions
        # ---------------------------------------------------------
        PromptUserMessage(
            content=[
                PromptTextContent(
                    text=(
                        "Yes, I'd like bullet points for the audio summary, "
                        "a short paragraph for the image description, and "
                        "the weather in NYC."
                    ),
                ),
            ],
        ),
    ],
    temperature=0.0,
)

## 4) Convert Prompt to Gemini Data & Make a Request

Next, we convert the `Prompt` to the Gemini-compatible dictionary using `convert_prompt_to_gemini_dict`.
We then pass that data into the model to get a response.

In [3]:
# Convert to Gemini data
gemini_data = convert_prompt_to_gemini_dict(prompt)

model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    system_instruction=gemini_data.system_instruction,
)

chat = model.start_chat(history=gemini_data.history)

# Send a user prompt or continue conversation
response = chat.send_message(
    gemini_data.message,
    generation_config=gemini_data.generation_config,
)

ip_display(Markdown(response.text))


Here's a summary of the audio, a description of the image, and the weather in NYC, formatted as requested:

**Audio Summary:**

* The audio is a discussion at a banking and financial services conference.
* The speakers discuss managing expenses and re-engineering processes to improve efficiency.
* They aim to eliminate waste and ensure that cost-cutting measures don't hinder growth.
* Expense management includes headcount reduction and cost management.
* Re-engineering involves business transformation, facilitated by a global platform.
* Cost savings are reinvested in revenue-generating opportunities.
* The company aims for 3-5% efficiency savings annually.
*  Significant investments (around $2.8 billion USD) have been made, partially offset by efficiency savings.
* The focus is shifting from initial investment to maximizing returns.
* The goal is to achieve positive operating leverage.


**Image Description:**

The photograph showcases a low-angle view of the Brooklyn Bridge in New York City on a bright, sunny day. The iconic bridge's massive stone towers and intricate network of cables dominate the foreground, set against a vivid blue sky punctuated by fluffy white clouds.  In the background, a portion of the Manhattan skyline and another bridge are visible, adding depth to the scene. The overall impression is one of grandeur and the powerful presence of this New York landmark.


**Weather in NYC:**

The weather in NYC is currently 72°F and partly cloudy.
